# APK Triage Lab  — Static Evidence Collection

**Configuration:** Static-only analysis · API vs open-source LLM comparison · Research + portfolio + practical tool.

This notebook creates an evidence-first APK triage pipeline. It hashes the APK, inspects its ZIP structure, parses Android metadata where possible, extracts DEX strings and network indicators, applies transparent review rules, and exports a JSON evidence bundle.

**Safety:** The APK is treated as untrusted data. This notebook does not install, launch, or execute it. Permissions and strings are indicators for investigation—not proof of malicious behavior.

## 0. Workflow

1. Upload one authorized APK.
2. Calculate SHA-256 and inspect the ZIP directory in memory.
3. Parse package metadata and manifest permissions with Androguard.
4. Extract readable DEX strings, URLs, domains, IPs, and selected API references.
5. Generate transparent heuristic flags.
6. Export a JSON evidence bundle.
7. Optionally submit identical evidence and prompts to an API model and an open-source model.

The LLM adapters are opt-in; no vendor or API key is required to run the static pipeline.

## 1. Install dependencies

In [14]:
%pip -q install androguard

## 2. Imports and safety limits

In [15]:
import hashlib, json, re, zipfile, io
from pathlib import Path

MAX_APK_BYTES = 200 * 1024 * 1024
MAX_ENTRY_BYTES = 100 * 1024 * 1024
MAX_STRINGS = 20000
MAX_EVIDENCE_STRINGS = 300
print("Ready. No APK has been executed.")

Ready. No APK has been executed.


## 3. Upload exactly one APK

In [16]:
from google.colab import files
uploaded = files.upload()
apk_names = [n for n in uploaded if n.lower().endswith(('.apk', '.xapk'))]
if len(apk_names) != 1:
    raise ValueError(f"Upload exactly one APK or XAPK; found {len(apk_names)}")
apk_name = apk_names[0]
apk_bytes = uploaded[apk_name]
if len(apk_bytes) > MAX_APK_BYTES:
    raise ValueError("APK exceeds the 200 MB notebook limit.")
print(apk_name, f"{len(apk_bytes):,} bytes")

Saving Chess_2.9.8_APKPure.xapk to Chess_2.9.8_APKPure (3).xapk
Chess_2.9.8_APKPure (3).xapk 4,124,725 bytes


## 4. File identity

In [17]:
sha256 = hashlib.sha256(apk_bytes).hexdigest()
file_identity = {"filename": apk_name, "size_bytes": len(apk_bytes), "sha256": sha256}
file_identity

{'filename': 'Chess_2.9.8_APKPure (3).xapk',
 'size_bytes': 4124725,
 'sha256': '620128a4f455ce934ddb12382f00182e8ded4e02890cccfb7419268f2b53a28b'}

## 5. Inspect ZIP container in memory

In [18]:
with zipfile.ZipFile(io.BytesIO(apk_bytes), "r") as zf:
    bad = zf.testzip()
    if bad:
        raise ValueError(f"ZIP CRC check failed: {bad}")
    infos = zf.infolist()
    inventory = [{"name": x.filename, "size": x.file_size,
                  "compressed_size": x.compress_size} for x in infos]
    dex_names = [x.filename for x in infos
                 if re.fullmatch(r"classes(?:[2-9][0-9]*)?\.dex", x.filename)]
    has_manifest = any(x.filename == "AndroidManifest.xml" for x in infos)
print("Entries:", len(inventory), "DEX:", dex_names, "Manifest:", has_manifest)
for item in inventory[:20]: print(item)

Entries: 5 DEX: [] Manifest: False
{'name': 'com.jetstartgames.chess.apk', 'size': 3462417, 'compressed_size': 3462417}
{'name': 'icon.png', 'size': 242503, 'compressed_size': 242503}
{'name': 'config.en.apk', 'size': 16793, 'compressed_size': 16793}
{'name': 'config.mdpi.apk', 'size': 401548, 'compressed_size': 401548}
{'name': 'manifest.json', 'size': 830, 'compressed_size': 830}


## 6. Parse Android manifest and metadata

In [19]:
manifest_data = {
    "parser": "androguard", "parse_status": "not_attempted",
    "package": None, "version_name": None, "version_code": None,
    "min_sdk": None, "target_sdk": None, "permissions": [],
    "activities": [], "services": [], "receivers": [], "providers": [],
    "parser_error": None
}
try:
    from androguard.misc import AnalyzeAPK
    a, d_list, dx = AnalyzeAPK(apk_bytes)
    manifest_data.update({
        "parse_status": "success",
        "package": a.get_package(),
        "version_name": a.get_androidversion_name(),
        "version_code": a.get_androidversion_code(),
        "min_sdk": a.get_min_sdk_version(),
        "target_sdk": a.get_target_sdk_version(),
        "permissions": sorted(set(a.get_permissions() or [])),
        "activities": sorted(set(a.get_activities() or [])),
        "services": sorted(set(a.get_services() or [])),
        "receivers": sorted(set(a.get_receivers() or [])),
        "providers": sorted(set(a.get_providers() or []))
    })
except Exception as exc:
    manifest_data["parse_status"] = "failed"
    manifest_data["parser_error"] = f"{type(exc).__name__}: {str(exc)[:400]}"
print(json.dumps(manifest_data, indent=2, default=str))

2026-11-24 08:00:32.272 +00:00 | DEBUG    | androguard.misc:AnalyzeAPK:50 - AnalyzeAPK
2026-11-24 08:00:32.274 +00:00 | DEBUG    | androguard.misc:AnalyzeAPK:65 - Analysing without session


{
  "parser": "androguard",
  "parse_status": "failed",
  "package": null,
  "version_name": null,
  "version_code": null,
  "min_sdk": null,
  "target_sdk": null,
  "permissions": [],
  "activities": [],
  "services": [],
  "receivers": [],
  "providers": [],
  "parser_error": "ValueError: embedded null byte"
}


## 7. Extract printable strings from DEX files

In [20]:
all_strings, dex_read_errors = [], []
with zipfile.ZipFile(io.BytesIO(apk_bytes), "r") as zf:
    for name in dex_names:
        info = zf.getinfo(name)
        if info.file_size > MAX_ENTRY_BYTES:
            dex_read_errors.append(f"{name}: skipped, too large")
            continue
        try:
            raw = zf.read(name)
            all_strings.extend(x.decode("utf-8", errors="replace")
                               for x in re.findall(rb"[ -~]{5,}", raw))
        except Exception as exc:
            dex_read_errors.append(f"{name}: {type(exc).__name__}")
all_strings = list(dict.fromkeys(all_strings))[:MAX_STRINGS]
print("Strings:", len(all_strings), "Read notes:", dex_read_errors)
for s in all_strings[:40]: print("-", s[:220])

Strings: 0 Read notes: []


## 8. Extract network indicators and selected API references

In [21]:
joined_text = "\n".join(all_strings)
urls = sorted(set(re.findall(r'''https?://[^\s"'<>]{4,300}''', joined_text, re.I)))
ips = sorted(set(re.findall(r"\b(?:25[0-5]|2[0-4]\d|1?\d?\d)(?:\.(?:25[0-5]|2[0-4]\d|1?\d?\d)){3}\b", joined_text)))
domains = sorted(set(re.findall(r"\b(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+[A-Za-z]{2,}\b", joined_text)))
api_keywords = ["DexClassLoader", "PathClassLoader", "Runtime;->exec",
                "ProcessBuilder", "getDeviceId", "getImei", "SmsManager",
                "sendTextMessage", "RECORD_AUDIO", "AccessibilityService",
                "requestInstallPackages", "Cipher", "SecretKeySpec", "WebView", "loadUrl"]
api_string_hits = {k: [s[:220] for s in all_strings if k.lower() in s.lower()][:20]
                   for k in api_keywords}
api_string_hits = {k:v for k,v in api_string_hits.items() if v}
network_indicators = {"urls": urls[:500], "domains": domains[:500], "ipv4_addresses": ips[:500]}
print("URLs:", len(urls), "Domains:", len(domains), "IPv4:", len(ips))
print(json.dumps(network_indicators, indent=2))
print(json.dumps(api_string_hits, indent=2))

URLs: 0 Domains: 0 IPv4: 0
{
  "urls": [],
  "domains": [],
  "ipv4_addresses": []
}
{}


## 9. Transparent heuristic review (not a malware verdict)

In [22]:
permission_groups = {
    "sms_access": ["SEND_SMS", "RECEIVE_SMS", "READ_SMS"],
    "microphone": ["RECORD_AUDIO"],
    "location": ["ACCESS_FINE_LOCATION", "ACCESS_COARSE_LOCATION", "ACCESS_BACKGROUND_LOCATION"],
    "contacts": ["READ_CONTACTS", "WRITE_CONTACTS"],
    "call_log": ["READ_CALL_LOG", "WRITE_CALL_LOG"],
    "camera": ["CAMERA"],
    "package_installation": ["REQUEST_INSTALL_PACKAGES"],
    "accessibility": ["BIND_ACCESSIBILITY_SERVICE"]
}
permission_flags = {}
for label, needles in permission_groups.items():
    hits = [p for p in manifest_data["permissions"]
            if any(n in p.upper() for n in needles)]
    if hits: permission_flags[label] = hits

flags = []
def add_flag(name, evidence, reason):
    flags.append({"flag": name, "evidence": evidence, "reason": reason})

if permission_flags.get("sms_access"):
    add_flag("sms_capability", permission_flags["sms_access"],
             "SMS permission declared; inspect actual use and app purpose.")
if permission_flags.get("microphone"):
    add_flag("microphone_capability", permission_flags["microphone"],
             "Microphone permission declared; verify the user-facing feature.")
if permission_flags.get("package_installation"):
    add_flag("package_installation_capability", permission_flags["package_installation"],
             "Can request package installation; inspect the user flow.")
if "DexClassLoader" in api_string_hits or "PathClassLoader" in api_string_hits:
    add_flag("dynamic_code_loading_reference", ["DEX string reference"],
             "Determine whether code loading is used and where code originates.")
if "SmsManager" in api_string_hits or "sendTextMessage" in api_string_hits:
    add_flag("sms_api_reference", ["DEX string reference"],
             "SMS API reference found; string presence does not prove execution.")
if urls:
    add_flag("network_endpoint_strings", urls[:20],
             "Classify endpoint ownership, reputation, and runtime use.")

baseline = {
    "assessment": "REVIEW_REQUIRED" if flags else "NO_RULE_TRIGGERED",
    "rule_flags": flags,
    "limitations": [
        "Not a malware classifier or probability.",
        "Permissions and strings do not prove behavior.",
        "No call graph, control-flow, certificate reputation, or dynamic analysis.",
        "No triggered rule does not mean the APK is safe."
    ]
}
print(json.dumps(baseline, indent=2))

{
  "assessment": "NO_RULE_TRIGGERED",
  "rule_flags": [],
  "limitations": [
    "Not a malware classifier or probability.",
    "Permissions and strings do not prove behavior.",
    "No call graph, control-flow, certificate reputation, or dynamic analysis.",
    "No triggered rule does not mean the APK is safe."
  ]
}


## 10. Build a structured evidence bundle

In [23]:
evidence_bundle = {
    "schema_version": "1.0",
    "file": file_identity,
    "archive": {"entry_count": len(inventory), "dex_files": dex_names,
                "manifest_present": has_manifest, "entries_sample": inventory[:100]},
    "android_metadata": manifest_data,
    "network_indicators": network_indicators,
    "api_reference_hits": api_string_hits,
    "sample_strings": all_strings[:MAX_EVIDENCE_STRINGS],
    "baseline_review": baseline,
    "analysis_scope": "static_only_no_install_no_execution"
}
evidence_json = json.dumps(evidence_bundle, indent=2, default=str)
print(evidence_json[:10000])
print("Bundle characters:", len(evidence_json))

{
  "schema_version": "1.0",
  "file": {
    "filename": "Chess_2.9.8_APKPure (3).xapk",
    "size_bytes": 4124725,
    "sha256": "620128a4f455ce934ddb12382f00182e8ded4e02890cccfb7419268f2b53a28b"
  },
  "archive": {
    "entry_count": 5,
    "dex_files": [],
    "manifest_present": false,
    "entries_sample": [
      {
        "name": "com.jetstartgames.chess.apk",
        "size": 3462417,
        "compressed_size": 3462417
      },
      {
        "name": "icon.png",
        "size": 242503,
        "compressed_size": 242503
      },
      {
        "name": "config.en.apk",
        "size": 16793,
        "compressed_size": 16793
      },
      {
        "name": "config.mdpi.apk",
        "size": 401548,
        "compressed_size": 401548
      },
      {
        "name": "manifest.json",
        "size": 830,
        "compressed_size": 830
      }
    ]
  },
  "android_metadata": {
    "parser": "androguard",
    "parse_status": "failed",
    "package": null,
    "version_name": null,
 

## 11. Export JSON evidence

In [24]:
evidence_path = Path("apk_evidence_bundle.json")
evidence_path.write_text(evidence_json, encoding="utf-8")
from google.colab import files
files.download(str(evidence_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Shared LLM prompt

For a fair comparison, submit the exact same evidence and prompt to both models. Keep API keys in Colab Secrets/environment variables; never commit them to GitHub. This prompt instructs the model to separate observed facts from hypotheses.

In [25]:
SYSTEM_PROMPT = """You are an Android application security triage assistant.
Use only the supplied JSON evidence. Do not claim a permission or string proves
malicious behavior. Separate observed facts from hypotheses. Do not invent evidence.
If evidence is insufficient, say so. Return JSON with:
summary, observed_facts, hypotheses, benign_explanations, follow_up_checks,
confidence (low/medium/high), limitations.
Do not give a definitive malware verdict from this static bundle alone."""

USER_PROMPT = (
    "Analyze this static APK evidence. Cite exact fields or values for important "
    "observations. Separate facts from hypotheses.\n\n" + evidence_json[:50000]
)
print("Prompt ready; characters:", len(USER_PROMPT))

Prompt ready; characters: 1922


### 12A. API model adapter (optional; OpenAI-compatible example)

In [26]:
# OPTIONAL: securely set OPENAI_API_KEY before running.
# %pip -q install openai
# import os
# from openai import OpenAI
# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# response = client.chat.completions.create(
#     model="YOUR_MODEL_NAME",
#     messages=[{"role":"system","content":SYSTEM_PROMPT},
#               {"role":"user","content":USER_PROMPT}],
#     temperature=0
# )
# api_model_output = response.choices[0].message.content
# print(api_model_output)

### 12B. Open-source model adapter (optional; Hugging Face example)

In [27]:
# OPTIONAL: model loading may exceed free Colab RAM/VRAM.
# %pip -q install transformers accelerate
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM
# HF_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # example; verify current model/licence
# tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     HF_MODEL_ID, device_map="auto", torch_dtype="auto"
# )
# messages = [{"role":"system","content":SYSTEM_PROMPT},
#             {"role":"user","content":USER_PROMPT}]
# prompt_text = tokenizer.apply_chat_template(
#     messages, tokenize=False, add_generation_prompt=True
# )
# inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
# output_ids = model.generate(**inputs, max_new_tokens=700, do_sample=False)
# open_source_output = tokenizer.decode(
#     output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
# )
# print(open_source_output)

## 13. Record comparison results

In [28]:
# Paste outputs after running each adapter.
api_model_output = None
open_source_output = None

comparison_record = {
    "apk_sha256": sha256,
    "prompt_version": "apk-triage-prompt-v1",
    "api_model_id": "record exact model ID",
    "open_source_model_id": "record exact model ID",
    "api_output": api_model_output,
    "open_source_output": open_source_output,
    "human_review": {
        "invented_evidence": None,
        "missed_observations": None,
        "fact_hypothesis_separation": None,
        "follow_up_quality": None,
        "overall_notes": None
    }
}
print(json.dumps(comparison_record, indent=2))

{
  "apk_sha256": "620128a4f455ce934ddb12382f00182e8ded4e02890cccfb7419268f2b53a28b",
  "prompt_version": "apk-triage-prompt-v1",
  "api_model_id": "record exact model ID",
  "open_source_model_id": "record exact model ID",
  "api_output": null,
  "open_source_output": null,
  "human_review": {
    "invented_evidence": null,
    "missed_observations": null,
    "fact_hypothesis_separation": null,
    "follow_up_quality": null,
    "overall_notes": null
  }
}


## 14. Research evaluation rubric

| Metric | What to evaluate |
|---|---|
| Evidence precision | Are cited facts actually in the bundle? |
| Evidence coverage | Did the model notice important observed indicators? |
| Hallucination rate | Did it invent permissions, APIs, domains, or behavior? |
| Fact/hypothesis separation | Are uncertain interpretations labelled? |
| Follow-up usefulness | Are checks specific and technically sound? |
| Calibration | Does confidence match evidence strength? |
| Latency/cost | How long and how much did each run consume? |

Start with 20–50 legally usable APK samples, including benign apps and appropriately sourced research samples. Document label provenance and uncertainty. Do not send private/proprietary APKs to third-party services without authorization.

## 15. Limitations and next milestone

This is a first-pass static triage tool, not a full reverse-engineering suite. It does not yet provide JADX decompilation, call-graph analysis, native-library analysis, certificate reputation, full intent-filter review, obfuscation analysis, YARA integration, dynamic sandboxing, or a trained classifier.

**Next milestone:** deepen DEX/API analysis, curate a documented evaluation set, then compare the API and open-source LLM outputs using the same evidence and rubric.

- `REVIEW_REQUIRED` means investigate further—not “malware.”
- `NO_RULE_TRIGGERED` means only that these simple rules did not fire—not “safe.”
- Keep the SHA-256 with every report.
- Never execute suspicious APKs in this notebook or on your everyday machine.